# Platform Monitoring Queries

This notebook answers questions by analyzing pipeline execution metadata stored in the `pipeline_executions` Delta table. 

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("AnalyzePipelineMonitoring")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.session.timeZone", "UTC")

PIPELINE_EXECUTIONS_PATH = (
    "s3a://delta-data/metadata/pipeline_executions"
)

executions = (
    spark.read
    .format("delta")
    .load(PIPELINE_EXECUTIONS_PATH)
)

executions.createOrReplaceTempView("pipeline_executions")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 17:04:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/24 17:04:02 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Questions

### Which dataset fails validation most frequently?

In [2]:
spark.sql("""
    SELECT
        dataset,
        SUM(
            CASE
                WHEN COALESCE(validation_failure_count, 0) > 0
                THEN 1
                ELSE 0
            END
        ) AS executions_with_failures,
        SUM(
            COALESCE(validation_failure_count, 0)
        ) AS total_failed_rules
    FROM pipeline_executions
    WHERE stage = 'silver'
    GROUP BY dataset
    ORDER BY total_failed_rules DESC, executions_with_failures DESC
""").show(truncate=False)

26/09/24 17:04:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+------------------------+------------------+
|dataset    |executions_with_failures|total_failed_rules|
+-----------+------------------------+------------------+
|taxi_trips |1                       |15                |
|taxi_zones |1                       |2                 |
|air_quality|0                       |0                 |
|weather    |0                       |0                 |
+-----------+------------------------+------------------+



### Which dataset requires the longest processing time?

In [3]:
spark.sql("""
    SELECT
        stage,
        dataset,
        COUNT(*) AS execution_count,
        ROUND(AVG(duration_seconds), 3) AS average_duration_seconds,
        ROUND(MAX(duration_seconds), 3) AS longest_duration_seconds
    FROM pipeline_executions
    GROUP BY stage, dataset
    ORDER BY average_duration_seconds DESC
""").show(truncate=False)

+-----------------+--------------------------------+---------------+------------------------+------------------------+
|stage            |dataset                         |execution_count|average_duration_seconds|longest_duration_seconds|
+-----------------+--------------------------------+---------------+------------------------+------------------------+
|silver           |taxi_trips                      |1              |39.241                  |39.241                  |
|silver           |air_quality                     |1              |34.289                  |34.289                  |
|gold             |integrated_taxi_trips           |1              |30.067                  |30.067                  |
|bronze           |air_quality                     |1              |13.213                  |13.213                  |
|bronze           |taxi_zones                      |1              |9.888                   |9.888                   |
|bronze           |taxi_trips                   

### How many records were rejected during each execution?

In [4]:
spark.sql("""
    SELECT
        pipeline_run_id,
        execution_id,
        stage,
        dataset,
        rejected_records,
        started_at
    FROM pipeline_executions
    ORDER BY rejected_records DESC, started_at DESC
""").show(100, truncate=False)

+------------------------------------+------------------------------------+-----------------+--------------------------------+----------------+--------------------------+
|pipeline_run_id                     |execution_id                        |stage            |dataset                         |rejected_records|started_at                |
+------------------------------------+------------------------------------+-----------------+--------------------------------+----------------+--------------------------+
|5042a04c-06c3-4a85-af11-d20961dbc3c8|b4409827-a34f-43bc-9483-a058e2bea1d1|silver           |taxi_trips                      |136945          |2026-09-24 17:00:03.125267|
|5042a04c-06c3-4a85-af11-d20961dbc3c8|211b8c45-5739-47d9-9c11-c991fc354a18|analytics_refresh|zone_weather_demand_variation   |0               |2026-09-24 17:02:02.661124|
|5042a04c-06c3-4a85-af11-d20961dbc3c8|5d27a007-dcea-47cf-99c8-93a0c456773f|analytics_refresh|air_quality_vs_demand           |0               |20

### How has processing time changed over multiple executions?

In [5]:
spark.sql("""
    SELECT
        stage,
        dataset,
        pipeline_run_id,
        started_at,
        duration_seconds
    FROM pipeline_executions
    ORDER BY stage, dataset, started_at
""").show(100, truncate=False)

+-----------------+--------------------------------+------------------------------------+--------------------------+----------------+
|stage            |dataset                         |pipeline_run_id                     |started_at                |duration_seconds|
+-----------------+--------------------------------+------------------------------------+--------------------------+----------------+
|analytics_refresh|air_quality_vs_demand           |5042a04c-06c3-4a85-af11-d20961dbc3c8|2026-09-24 17:01:59.175022|3.307           |
|analytics_refresh|average_trip_distance_by_weather|5042a04c-06c3-4a85-af11-d20961dbc3c8|2026-09-24 17:01:55.758916|2.542           |
|analytics_refresh|monthly_zone_demand             |5042a04c-06c3-4a85-af11-d20961dbc3c8|2026-09-24 17:01:51.805677|3.77            |
|analytics_refresh|zone_weather_demand_variation   |5042a04c-06c3-4a85-af11-d20961dbc3c8|2026-09-24 17:02:02.661124|2.116           |
|bronze           |air_quality                     |5042a04c-0

In [6]:
spark.stop()